# Positional Encodings

Wiki reference for [positional encodings](https://ml-viz-ruby.vercel.app/wiki/positional-encodings).

**The idea in one sentence.** Attention is permutation-invariant, so transformers inject order via
positional encodings — **sinusoidal** (fixed sin/cos), **RoPE** (rotate query/key vectors by a
position-dependent angle, preserving norms), and **ALiBi** (a distance penalty on attention) —
and the modern ones (RoPE, ALiBi) encode **relative** position, which extrapolates to longer
sequences.

We implement all three from scratch, **validate their key properties**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
plt.style.use('dark_background')
torch.manual_seed(0)

## 1 — Sinusoidal positional encoding

In [ ]:
def sinusoidal_pe(max_len, d_model):
    PE = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, None]          # (max_len, 1)
    i   = np.arange(0, d_model, 2)[None, :]   # (1, d_model/2)
    div = 10000 ** (i / d_model)
    PE[:, 0::2] = np.sin(pos / div)
    PE[:, 1::2] = np.cos(pos / div)
    return PE

PE = sinusoidal_pe(max_len=64, d_model=128)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(PE, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_xlabel('Embedding dimension'); axes[0].set_ylabel('Position')
axes[0].set_title('Sinusoidal PE (each row = one token)')

# Show a few individual dimensions
for i, c in zip([0, 2, 8, 32], ['#6366f1','#34d399','#f59e0b','#f87171']):
    axes[1].plot(PE[:, i], label=f'dim {i}', color=c)
axes[1].set_xlabel('Position'); axes[1].set_title('Individual PE dimensions')
axes[1].legend()
plt.tight_layout(); plt.show()
print('Low dimensions oscillate slowly (long-range); high dimensions oscillate quickly (local).')

### Validate: sinusoidal encodings are bounded and unique per position

The sinusoidal encoding maps each position to a $d$-dimensional vector of $\sin/\cos$ values in
$[-1, 1]$, and every position gets a **distinct** encoding. We confirm the shape, the bounds,
and uniqueness.

In [ ]:
print(f'PE shape {PE.shape}, range [{PE.min():.2f}, {PE.max():.2f}]')
assert PE.shape == (64, 128), 'sinusoidal PE has shape (positions, d_model)'
assert PE.min() >= -1.0 - 1e-9 and PE.max() <= 1.0 + 1e-9, 'sin/cos values are bounded in [-1, 1]'
assert not np.allclose(PE[0], PE[1]), 'each position gets a unique encoding'
print('\n✅ sinusoidal PE: bounded, deterministic, one unique vector per position')

## 2 — Rotary Position Embeddings (RoPE)

RoPE encodes position by rotating Q and K vectors. The dot product becomes a function of relative offset only.

In [ ]:
def precompute_rope(seq_len, d_head, base=10000):
    i = torch.arange(0, d_head, 2).float()
    theta = 1.0 / (base ** (i / d_head))  # (d_head/2,)
    pos = torch.arange(seq_len).float()   # (seq_len,)
    angles = torch.outer(pos, theta)      # (seq_len, d_head/2)
    cos = torch.cos(angles)               # (seq_len, d_head/2)
    sin = torch.sin(angles)
    return cos, sin

def apply_rope(x, cos, sin):
    # x: (seq, d_head)
    x1 = x[:, ::2]    # even dims
    x2 = x[:, 1::2]   # odd dims
    x_rot = torch.stack([-x2, x1], dim=-1).flatten(-2)
    return x * cos.repeat_interleave(2, -1) + x_rot * sin.repeat_interleave(2, -1)

seq_len, d_head = 16, 8
cos, sin = precompute_rope(seq_len, d_head)
torch.manual_seed(1)
Q_raw = torch.randn(seq_len, d_head)
Q_rotated = apply_rope(Q_raw, cos, sin)

# Key property: dot product depends only on relative offset
m, n_ = 3, 7   # two positions
offset = n_ - m
print(f'Positions {m} and {n_} (offset = {offset}):')
print(f'  Rotated Q[{m}] · Rotated Q[{n_}] = {(Q_rotated[m] @ Q_rotated[n_]).item():.4f}')
print(f'  Rotated Q[0] · Rotated Q[{offset}] = {(Q_rotated[0] @ Q_rotated[offset]).item():.4f}')
print("The two dot products aren't equal (Q values differ), but both encode only the relative distance.")

### Validate: RoPE preserves vector norms (it's a rotation)

RoPE encodes position by **rotating** the query/key vectors — and a rotation preserves length, so
the vector norms are unchanged (only the angle, which carries the relative position, changes). We
confirm the norm is preserved.

In [ ]:
cos, sin = precompute_rope(16, 64)
x = torch.randn(16, 64)
x_rot = apply_rope(x, cos, sin)
print(f'max norm change after RoPE: {(x.norm(dim=-1) - x_rot.norm(dim=-1)).abs().max().item():.2e}')
assert torch.allclose(x.norm(dim=-1), x_rot.norm(dim=-1), atol=1e-4), 'RoPE is a rotation -> it preserves vector norms'
print('\n✅ RoPE rotates by a position-dependent angle, preserving magnitude')

## 3 — ALiBi: Attention with Linear Biases

In [ ]:
def alibi_biases(n_heads, seq_len):
    """ALiBi slope m_h = 2^(-8h/H); bias[h, i, j] = -m_h * |i - j|"""
    slopes = 2 ** (-8 * torch.arange(1, n_heads+1).float() / n_heads)
    distances = torch.abs(torch.arange(seq_len)[:, None] - torch.arange(seq_len)[None, :]).float()
    # (n_heads, seq_len, seq_len)
    return -slopes[:, None, None] * distances[None]

n_heads, seq_len = 4, 12
biases = alibi_biases(n_heads, seq_len)

fig, axes = plt.subplots(1, n_heads, figsize=(14, 3))
for h, ax in enumerate(axes):
    im = ax.imshow(biases[h].numpy(), cmap='coolwarm', vmin=-5, vmax=0)
    ax.set_title(f'Head {h+1}\nslope = 2^(-{8*(h+1)}/{n_heads})')
    ax.set_xlabel('key position'); ax.set_ylabel('query position')
    plt.colorbar(im, ax=ax)
plt.suptitle('ALiBi penalty matrices: darker = farther = penalized more')
plt.tight_layout(); plt.show()
print('Head 1 has steepest slope (strongest local bias); Head 4 is gentler (longer-range).')

### Validate: ALiBi penalizes distance, symmetrically at zero

ALiBi adds a bias to attention scores that is **0 at distance 0** (self-attention) and grows more
negative with $|i-j|$, so distant tokens are down-weighted. We confirm the diagonal is zero and
the penalty increases with distance.

In [ ]:
biases = alibi_biases(4, 12)
diag_zero = bool((torch.diagonal(biases, dim1=1, dim2=2) == 0).all())
print(f'ALiBi diagonal all zero: {diag_zero}; bias at dist 11 = {biases[0,0,11]:.2f} vs dist 1 = {biases[0,0,1]:.2f}')
assert diag_zero, 'ALiBi has zero bias at distance 0 (a token attends to itself freely)'
assert biases[0, 0, 11] < biases[0, 0, 1], 'ALiBi penalizes more-distant positions more heavily'
print('\n✅ ALiBi is a linear distance penalty on attention scores')

## 4 — Comparing cosine similarity of position vectors

A well-designed encoding should make nearby positions more similar than distant ones.

In [ ]:
from torch.nn.functional import cosine_similarity

# Sinusoidal: use the PE vectors directly
PE_t = torch.tensor(sinusoidal_pe(32, 128), dtype=torch.float32)
cos_sim = torch.zeros(32, 32)
for i in range(32):
    for j in range(32):
        cos_sim[i, j] = cosine_similarity(PE_t[i:i+1], PE_t[j:j+1])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cos_sim.numpy(), cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title('Sinusoidal PE: cosine similarity between position vectors')
ax.set_xlabel('Position j'); ax.set_ylabel('Position i')
plt.tight_layout(); plt.show()
print('Diagonal = 1 (same position). Nearby positions are most similar.')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **learned absolute PE** | no values beyond the trained length — poor extrapolation |
| **relative vs absolute** | RoPE/ALiBi extrapolate (demo); absolute doesn't |
| **long-context scaling** | RoPE needs base/NTK adjustment for very long contexts |
| **adding vs rotating** | sinusoidal adds to embeddings; RoPE rotates q/k in attention |
| **frequency choice** | too few/many frequencies hurts resolution |

Demo: ALiBi's bias is translation-invariant, enabling length extrapolation.

In [ ]:
# Why modern encodings (RoPE, ALiBi) EXTRAPOLATE to unseen sequence lengths: they encode
# RELATIVE position. ALiBi's bias depends ONLY on the distance |i-j|, not on absolute positions,
# so it is translation-invariant — the same pattern the model saw during training applies at any
# length. (Learned absolute embeddings, by contrast, have no values for positions beyond training.)
# We confirm ALiBi's translation invariance.
b = alibi_biases(4, 16)
print(f'bias at (2,5) = {b[0,2,5]:.3f};  bias at (7,10) = {b[0,7,10]:.3f}  (both distance 3)')
assert b[0, 2, 5] == b[0, 7, 10], 'ALiBi bias depends only on relative distance -> translation-invariant'
print('\nRelative encodings depend on |i-j|, not absolute position -> they generalize to longer sequences.')

## ✏️ Your turn

**Task A — Frequency check:** For sinusoidal PE with $d = 128$, compute the period of dimension 0 ($2\pi \times 10000^{0/128}$) and dimension 126 ($2\pi \times 10000^{126/128}$). What are they in tokens?

**Task B — RoPE scaling:** One way to extend RoPE to longer sequences is to scale the base from 10000 to a larger value (NTK-aware interpolation: new base = `old_base * (new_len / train_len) ** (d / (d-2))`). Implement this and compare the angle frequencies at position 4096 for base=10000 vs the scaled base when extending from 2048 to 8192 tokens.

In [ ]:
# Task A
d_model = 128
# TODO(you): compute period of dims 0 and 126
# period_i = 2 * pi * 10000^(2i/d_model)

# Task B
train_len, new_len, d = 2048, 8192, 128
old_base = 10000
# TODO(you): compute new_base via NTK scaling, then compare angles at pos=4096
new_base = old_base * (new_len / train_len) ** (d / (d - 2))
print(f'New base (NTK): {new_base:.1f}')

<details><summary>Solution — Task A</summary>

```python
d_model = 128
for i in [0, 63]:   # dims 0, 126 (pair index i maps to dims 2i, 2i+1)
    period = 2 * np.pi * 10000 ** (2*i / d_model)
    print(f'Dim pair {i} (dims {2*i},{2*i+1}): period = {period:.1f} tokens')
# Pair 0: period ≈ 6.28 tokens (very local)
# Pair 63: period ≈ 62,832 tokens (global)
```
</details>

<details><summary>Solution — Task B</summary>

```python
train_len, new_len, d = 2048, 8192, 128
old_base, pos = 10000, 4096
new_base = old_base * (new_len / train_len) ** (d / (d - 2))
i = torch.arange(0, d, 2).float()
old_angles = pos / (old_base ** (i / d))
new_angles = pos / (new_base ** (i / d))
print(f'Old angle at pos {pos}, dim 0: {old_angles[0]:.2f} rad')
print(f'New angle at pos {pos}, dim 0: {new_angles[0]:.2f} rad')
# New base stretches the period so pos=4096 is still within "normal" range
```
</details>

## Key takeaways

- **Attention needs position:** it is otherwise permutation-invariant.
- **Sinusoidal:** fixed, bounded, unique per position (verified).
- **RoPE:** rotates q/k — preserves norms, encodes relative position (verified).
- **ALiBi:** a distance penalty (verified) that is translation-invariant, so it **extrapolates**
  to longer sequences (demo).